In [ ]:
import pathlib
import subprocess as sp
import numpy as np
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-colorblind')

In [ ]:
import yaml

base_path = Path('../../../').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Load RUSH h5ad

In [ ]:
RUSH_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_proc/240124_PsychAD_freeze3_RUSH_clean.h5ad'

In [ ]:
dat_rush = singlecell_utils.read_everything_but_X(RUSH_H5AD_PATH)

In [ ]:
dat_rush.obs[['Channel', 'n_counts']]

In [ ]:
# Get total counts per channel

gb_chn = dat_rush.obs.groupby('Channel', observed=True)

se_cnt_per_chn = gb_chn.n_counts.sum()
se_cnt_per_chn

In [ ]:
se_subid = gb_chn.SubID.first() 
se_rep = gb_chn.rep.first()

# Find ratio between gene mapped counts vs total bam file size

In [ ]:
# n_counts in h5ad represents gene mapped read counts.
# bam files in below path contains all reads with freeze3 filtered cell barcodes.

FILTERED_BAM_PATH = '/sc/arion/projects/CommonMind/yeon/p/APA/split_per_donor_and_count/RUSH/pl/count/{}/rep_{}.assigned.sorted.bam'

In [ ]:
print(len(se_cnt_per_chn))

In [ ]:
# Select random 15; get read counts


ncounts_in_h5ad = []
nlines_in_bam = []

for idx in [7, 15, 40, 55, 77, 99, 120, 136, 154, 180, 210, 222, 240, 268, 281]:
    break
    test_chn = se_cnt_per_chn.index[idx]
    ncounts_in_h5ad.append(se_cnt_per_chn.loc[test_chn])

    subid = se_subid.loc[test_chn]
    rep = se_rep.loc[test_chn]
    filt_bam_path = FILTERED_BAM_PATH.format(subid, rep)

    cmd = f'ml samtools; samtools view -@ 10 -c {filt_bam_path} '
    print(cmd)

    result = sp.run(cmd, stdout=sp.PIPE, stderr=sp.PIPE, shell=True)
    nlines_in_bam.append(int(result.stdout))

In [ ]:
# Bam size / n_counts ratio 

np.array(nlines_in_bam) / np.array(ncounts_in_h5ad)

## Conclusion: ratio range : 2-10

# Read full PsychAD H5AD

In [ ]:
FULL_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_proc/240124_PsychAD_freeze3_FULL_clean.h5ad'
dat_full = singlecell_utils.read_everything_but_X(FULL_H5AD_PATH)

In [ ]:
# Free some memories
del dat_rush 
del se_cnt_per_chn
del se_rep
del se_subid

# Class level subsampling

In [ ]:
df_obs = dat_full.obs.reset_index()
df_obs['cell_barcode'] = 'CB:Z:' + df_obs['barcodekey'].apply(lambda x: x.split('-')[2])

gb_class = df_obs.groupby('class', observed=True)

In [ ]:
gb_class.cell_barcode.count()

In [ ]:
RANDOM_SEED = 1
NUM_CELLS_PER_CLASS = 20000
OUTDIR_CLASS = pathlib.Path(f'class_level_{NUM_CELLS_PER_CLASS}_cells')
OUTDIR_CLASS.mkdir(parents=True, exist_ok=True)

df_bc_ss_class = gb_class.sample(n=NUM_CELLS_PER_CLASS, random_state=RANDOM_SEED)
Counter(df_bc_ss_class['class'])

In [ ]:
# Gene mapped readcounts per class
se_bc_ss_class_ncounts = df_bc_ss_class.groupby('class', observed=True).n_counts.sum()
se_bc_ss_class_ncounts

In [ ]:
print('{:,}'.format(se_bc_ss_class_ncounts.min() * 2))
print('{:,}'.format(se_bc_ss_class_ncounts.min() * 10))
print('{:,}'.format(se_bc_ss_class_ncounts.max() * 2))
print('{:,}'.format(se_bc_ss_class_ncounts.max() * 10))

## Conclusion: Merging 5 billion reads is possible, so do the subsampling after getting all the reads.

In [ ]:
print(df_bc_ss_class.Channel.nunique(),
      df_obs.Channel.nunique(),
      100 * df_bc_ss_class.Channel.nunique() / df_obs.Channel.nunique())

print(df_bc_ss_class.SubID.nunique(),
      df_obs.SubID.nunique(),
      100 * df_bc_ss_class.SubID.nunique() / df_obs.SubID.nunique())

In [ ]:
df_bc_ss_class[['poolID', 'round_num']]

In [ ]:
#Counter(df_bc_ss_class['SubID'])
dat_full.obs['SubID'].nunique()

In [ ]:
df_bc_ss_class.poolID.nunique()

## Not every pool is choosen for all classes, so keep pool and round info

In [ ]:
df_ss_class_files = df_bc_ss_class[['round_num', 'poolID', 'class']].drop_duplicates().sort_values(by=['round_num', 'poolID', 'class'])
df_ss_class_files.to_csv(OUTDIR_CLASS / 'subsampled_poolIDs.tsv', sep='\t', index=False)
df_ss_class_files

## Check subsampling coverage - class level

In [ ]:
Counter(Counter(df_ss_class_files['poolID']).values())

In [ ]:
df_ss_class_nuniq = df_bc_ss_class.groupby('class', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_class_nuniq

In [ ]:
df_ss_class_coverage = 100 * df_ss_class_nuniq / df_obs.groupby('class', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_class_coverage

In [ ]:
fig = plt.figure(figsize=(4, 3))
ax = fig.add_subplot()
df_ss_class_coverage.plot(kind='bar', width=0.7, ax=ax, zorder=2)
ax.set_ylabel('Coverage (%)')
ax.set_ylim([0, 100])
ax.set_xlabel('Class')
ax.legend(loc=(1.01, 0.0))
ax.yaxis.grid(zorder=1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
ax.set_title(f'Subsampling coverage\n(# cells = {NUM_CELLS_PER_CLASS})')
plt.show()

## Write barcodes for class level

In [ ]:
CLASS_GREP_PATH = OUTDIR_CLASS / 'barcodes_greplist'
pathlib.Path(CLASS_GREP_PATH).mkdir(exist_ok=True)

CLASS_WITHCLASS_PATH = OUTDIR_CLASS / 'barcodes_withclass'
pathlib.Path(CLASS_WITHCLASS_PATH).mkdir(exist_ok=True)

In [ ]:
for pi, df in df_bc_ss_class.groupby('poolID', observed=True):
    round_num = df.round_num.iloc[0]

    grep_file_name = CLASS_GREP_PATH / f'round{round_num}_{pi}_greplist.txt'
    class_file_name = CLASS_WITHCLASS_PATH / f'round{round_num}_{pi}_cell_barcodes_class.tsv'

    df[['cell_barcode']].to_csv(grep_file_name, index=False, header=False)
    df[['cell_barcode', 'class']].to_csv(class_file_name, index=False, header=False, sep='\t')



# Subsample by subclass

In [ ]:
pd.Series(Counter(df_obs.subclass)).sort_values()

In [ ]:
RANDOM_SEED = 1
NUM_CELLS_PER_SUBCLASS = 20000
OUTDIR_SUBCLASS = pathlib.Path(f'subclass_level_{NUM_CELLS_PER_SUBCLASS}_cells')
OUTDIR_SUBCLASS.mkdir(exist_ok=True)

# remove cells below subsampled counts
se_subclass_cell_cnt = df_obs.groupby('subclass', observed=True).barcodekey.count()
below_count_subclasses = set(se_subclass_cell_cnt[se_subclass_cell_cnt < NUM_CELLS_PER_SUBCLASS].index)

print('These subclasses are removed due to low cell counts: ')
print(below_count_subclasses)

df_obs_cut = df_obs[~df_obs.subclass.isin(below_count_subclasses)]
print(len(df_obs), len(df_obs_cut))

gb_subclass = df_obs_cut.groupby('subclass', observed=True)
df_bc_ss_subclass = gb_subclass.sample(n=NUM_CELLS_PER_SUBCLASS, random_state=RANDOM_SEED)
Counter(df_bc_ss_subclass['subclass'])

In [ ]:
# Gene mapped readcounts per subclass
# ~1/5 subsampling is required for EN
se_bc_ss_subclass_ncounts = df_bc_ss_subclass.groupby('subclass', observed=True).n_counts.sum()
se_bc_ss_subclass_ncounts

## Not every pool is choosen for all classes, so keep pool and round info

In [ ]:
df_ss_subclass_files = df_bc_ss_subclass[['round_num', 'poolID', 'subclass']].drop_duplicates().sort_values(by=['round_num', 'poolID', 'subclass'])
df_ss_subclass_files.to_csv(OUTDIR_SUBCLASS / 'subsampled_poolIDs.tsv', sep='\t', index=False)
df_ss_subclass_files

## Check subsampling coverage - subclass level

In [ ]:
Counter(Counter(df_ss_subclass_files['poolID']).values())

In [ ]:
df_ss_subclass_nuniq = df_bc_ss_subclass.groupby('subclass', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_subclass_nuniq

In [ ]:
df_ss_subclass_coverage = 100 * df_ss_subclass_nuniq / df_obs_cut.groupby('subclass', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_subclass_coverage

In [ ]:
fig = plt.figure(figsize=(10, 3))
ax = fig.add_subplot()
df_ss_subclass_coverage.plot(kind='bar', width=0.7, zorder=10, ax=ax)
ax.set_ylabel('Coverage (%)')
ax.set_xlabel('Subclass')
ax.set_ylim([0, 100])
ax.legend(loc=(1.01, 0.0))
ax.yaxis.grid(zorder=1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
ax.set_title(f'Subsampling coverage (# cells = {NUM_CELLS_PER_SUBCLASS})')
plt.show()

## Write barcodes for subclass level

In [ ]:
SUBCLASS_GREP_PATH = OUTDIR_SUBCLASS / 'barcodes_greplist'
pathlib.Path(SUBCLASS_GREP_PATH).mkdir(exist_ok=True)

SUBCLASS_WITHSUBCLASS_PATH = OUTDIR_SUBCLASS / 'barcodes_withsubclass'
pathlib.Path(SUBCLASS_WITHSUBCLASS_PATH).mkdir(exist_ok=True)

In [ ]:
for pi, df in df_bc_ss_subclass.groupby('poolID', observed=True):
    round_num = df.round_num.iloc[0]
    
    grep_file_name = SUBCLASS_GREP_PATH / f'round{round_num}_{pi}_greplist.txt'
    subclass_file_name = SUBCLASS_WITHSUBCLASS_PATH / f'round{round_num}_{pi}_cell_barcodes_subclass.tsv'
    
    df[['cell_barcode']].to_csv(grep_file_name, index=False, header=False)
    df[['cell_barcode', 'subclass']].to_csv(subclass_file_name, index=False, header=False, sep='\t')
    
    

### Validation: open one of them

In [ ]:
print(grep_file_name)
pd.read_csv(grep_file_name).head()

In [ ]:
print(subclass_file_name)
df = pd.read_table(subclass_file_name, header=None)
print(Counter(df[1]))
df.head()

# Subsample by Age X class

In [ ]:
AGE_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_rc/h5ad_final/AGING_2024-02-01_22_23.h5ad'
dat_age = singlecell_utils.read_everything_but_X(AGE_H5AD_PATH)

In [ ]:
pd.crosstab(dat_age.obs.r10x, dat_age.obs['class'])


In [ ]:
# Each SubID has unique age
dat_age.obs.groupby('SubID', observed=True).r10x.nunique().max()

In [ ]:
se_ages = dat_age.obs.groupby('SubID', observed=True).r10x.first()
subid_to_age = dict(zip(se_ages.index, se_ages))

## Add age info to full h5ad table

In [ ]:
df_obs['r10x'] = df_obs.SubID.map(subid_to_age)

In [ ]:
df_cross = pd.crosstab(df_obs.r10x, df_obs['class'])
df_cross

In [ ]:
# Conclusion: They are the same
df_cross - pd.crosstab(dat_age.obs.r10x, dat_age.obs['class'])

## Remove ages or cell types with small number of cells

In [ ]:
RANDOM_SEED = 1
NUM_CELLS_PER_AGE_CLASS = 5000
OUTDIR_AGE_CLASS = pathlib.Path(f'age_class_level_{NUM_CELLS_PER_AGE_CLASS}_cells')
OUTDIR_AGE_CLASS.mkdir(exist_ok=True)


In [ ]:
df_cross.mean(), df_cross.mean(axis=1)

In [ ]:
df_cross >= NUM_CELLS_PER_AGE_CLASS

In [ ]:
# Remove Endo, Mural, and Infancy

removed_class = ['Endo', 'Mural']
removed_age = ['Infancy']

In [ ]:
df_obs_agecut = df_obs[
                    ~(df_obs['class'].isin(removed_class) | 
                      df_obs.r10x.isin(removed_age) |
                      df_obs.r10x.isnull())
                    ].copy()
pd.crosstab(df_obs_agecut.r10x, df_obs_agecut['class'])

In [ ]:
df_obs_agecut['age_class'] = df_obs_agecut.r10x.astype(str) + '-' + df_obs_agecut['class'].astype(str)

In [ ]:
gb_age_class = df_obs_agecut.groupby('age_class', observed=True)
df_bc_ss_age_class = gb_age_class.sample(n=NUM_CELLS_PER_AGE_CLASS, random_state=RANDOM_SEED)
Counter(df_bc_ss_age_class['age_class'])

In [ ]:
# Gene mapped readcounts per age X class
# ~1/5 subsampling is required for EN
se_bc_ss_age_class_ncounts = df_bc_ss_age_class.groupby('age_class', observed=True).n_counts.sum()
se_bc_ss_age_class_ncounts.sort_values()

## Not every pool is choosen for all age X classes, so keep pool and round info

In [ ]:
df_ss_age_class_files = df_bc_ss_age_class[['round_num', 'poolID', 'age_class']
                                           ].drop_duplicates().sort_values(by=['round_num', 'poolID', 'age_class'])
df_ss_age_class_files.to_csv(OUTDIR_AGE_CLASS / 'subsampled_poolIDs.tsv', sep='\t', index=False)
df_ss_age_class_files

## Check subsampling coverage - age x class

In [ ]:
Counter(Counter(df_ss_age_class_files['poolID']).values())

In [ ]:
df_ss_age_class_nuniq = df_bc_ss_age_class.groupby('age_class', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_age_class_nuniq 

In [ ]:
df_ss_age_class_coverage = 100 * df_ss_age_class_nuniq / df_obs_agecut.groupby('age_class', observed=True
                                                                              )[['barcodekey', 'SubID', 'poolID']].nunique()

In [ ]:
# Sort dataframe
age_orders = 'Childhood Adolescence Late_adult Middle_adult Young_adult'.split()
age_orders = dict(zip(age_orders, range(len(age_orders))))
class_orders = 'IN EN Oligo OPC Astro Immune'.split()
class_orders = dict(zip(class_orders, range(len(class_orders))))

df_ss_age_class_coverage['age_order'] = df_ss_age_class_coverage.index.to_series().apply(lambda x: x.split('-')[0]).map(age_orders)
df_ss_age_class_coverage['class_order'] = df_ss_age_class_coverage.index.to_series().apply(lambda x: x.split('-')[1]).map(class_orders)

df_ss_age_class_coverage.sort_values(by=['age_order', 'class_order'], inplace=True)
del df_ss_age_class_coverage['age_order'] 
del df_ss_age_class_coverage['class_order'] 

In [ ]:
fig = plt.figure(figsize=(9, 3))
ax = fig.add_subplot()
df_ss_age_class_coverage.plot(kind='bar', width=0.7, zorder=10, ax=ax)
ax.set_ylabel('Coverage (%)')
ax.set_xlabel('r10X and class')
ax.set_ylim([0, 102])
ax.legend(loc=(1.01, 0.0))
ax.yaxis.grid(zorder=1)
tick_labels = [xt.get_text().replace('_', ' ').replace('-', ' - ') for xt in ax.get_xticklabels()]
ax.set_xticklabels(tick_labels, rotation=35, ha='right', rotation_mode='anchor')
ax.set_title(f'Subsampling coverage (# cells per group = {NUM_CELLS_PER_AGE_CLASS})')
plt.savefig(f'Subsampling_coverage_ageXclass_{NUM_CELLS_PER_AGE_CLASS}cells.pdf', bbox_inches='tight')

plt.show()

In [ ]:
df_ss_age_class_coverage

## Write barcodes for age X class level

In [ ]:
AGE_CLASS_GREP_PATH = OUTDIR_AGE_CLASS / 'barcodes_greplist'
pathlib.Path(AGE_CLASS_GREP_PATH).mkdir(exist_ok=True)

AGE_CLASS_WITH_AGE_CLASS_PATH = OUTDIR_AGE_CLASS / 'barcodes_with_age_class'
pathlib.Path(AGE_CLASS_WITH_AGE_CLASS_PATH).mkdir(exist_ok=True)

In [ ]:
for pi, df in df_bc_ss_age_class.groupby('poolID', observed=True):
    round_num = df.round_num.iloc[0]
    
    grep_file_name = AGE_CLASS_GREP_PATH / f'round{round_num}_{pi}_greplist.txt'
    age_class_file_name = AGE_CLASS_WITH_AGE_CLASS_PATH / f'round{round_num}_{pi}_cell_barcodes_age_class.tsv'
    
    df[['cell_barcode']].to_csv(grep_file_name, index=False, header=False)
    df[['cell_barcode', 'age_class']].to_csv(age_class_file_name, index=False, header=False, sep='\t')
    
    

### Validation: open one of them

In [ ]:
print(grep_file_name)
pd.read_csv(grep_file_name).head()

In [ ]:
print(age_class_file_name)
df = pd.read_table(age_class_file_name, header=None)
print(Counter(df[1]))
df.head()

# Subsample 5k Endo Mural cells, since they are removed in age X class

In [ ]:
NUM_CELLS_PER_RMCLASS = NUM_CELLS_PER_AGE_CLASS
OUTDIR_RMCLASS = pathlib.Path(f'rmclass_level_{NUM_CELLS_PER_RMCLASS}_cells')
OUTDIR_RMCLASS.mkdir(exist_ok=True)


In [ ]:
len(df_obs), removed_class

In [ ]:
gb_rmclass = df_obs[df_obs['class'].isin(removed_class)].groupby('class', observed=True)
df_bc_ss_rmclass = gb_rmclass.sample(n=NUM_CELLS_PER_RMCLASS, random_state=RANDOM_SEED)
Counter(df_bc_ss_rmclass['subclass'])

## Not every pool is choosen for all classes, so keep pool and round info

In [ ]:
df_ss_rmclass_files = df_bc_ss_rmclass[['round_num', 'poolID', 'class']].drop_duplicates().sort_values(by=['round_num', 'poolID', 'class'])
df_ss_rmclass_files.to_csv(OUTDIR_RMCLASS / 'subsampled_poolIDs.tsv', sep='\t', index=False)
df_ss_rmclass_files

## Check subsampling coverage - rmclass level

In [ ]:
Counter(Counter(df_ss_rmclass_files['poolID']).values())

In [ ]:
df_ss_rmclass_nuniq = df_bc_ss_rmclass.groupby('class', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_rmclass_nuniq

In [ ]:
df_ss_rmclass_coverage = 100 * df_ss_rmclass_nuniq / \
            df_obs[df_obs['class'].isin(removed_class)].groupby('class', observed=True)[['barcodekey', 'SubID', 'poolID']].nunique()
df_ss_rmclass_coverage

In [ ]:
fig = plt.figure(figsize=(1.2, 3))
ax = fig.add_subplot()
df_ss_rmclass_coverage.plot(kind='bar', width=0.7, zorder=10, ax=ax)
ax.set_ylabel('Coverage (%)')
ax.set_xlabel('class')
ax.set_ylim([0, 100])
ax.legend(loc=(1.01, 0.0))
ax.yaxis.grid(zorder=1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
ax.set_title(f'Subsampling coverage (# cells = {NUM_CELLS_PER_RMCLASS})\n')
plt.show()

## Write barcodes

In [ ]:
RMCLASS_GREP_PATH = OUTDIR_RMCLASS / 'barcodes_greplist'
pathlib.Path(RMCLASS_GREP_PATH).mkdir(exist_ok=True)

RMCLASS_WITH_RMCLASS_PATH = OUTDIR_RMCLASS / 'barcodes_with_rmclass'
pathlib.Path(RMCLASS_WITH_RMCLASS_PATH).mkdir(exist_ok=True)

for pi, df in df_bc_ss_rmclass.groupby('poolID', observed=True):
    round_num = df.round_num.iloc[0]
    
    grep_file_name = RMCLASS_GREP_PATH / f'round{round_num}_{pi}_greplist.txt'
    rmclass_file_name = RMCLASS_WITH_RMCLASS_PATH / f'round{round_num}_{pi}_cell_barcodes_rmclass.tsv'
    
    df[['cell_barcode']].to_csv(grep_file_name, index=False, header=False)
    df[['cell_barcode', 'class']].to_csv(rmclass_file_name, index=False, header=False, sep='\t')
    
    

### Validation: open one of them

In [ ]:
print(grep_file_name)
pd.read_csv(grep_file_name).head()

In [ ]:
print(rmclass_file_name)
df = pd.read_table(rmclass_file_name, header=None)
print(Counter(df[1]))
df.head()